# 01 · Extract — Peras / PostHog

**Purpose:** Reproducibly pull scoped event data from Peras's PostHog project via the
Query API (HogQL), inventory what exists, and land a raw extract locally for cleaning.

**Safety contract for this notebook**
- Credentials are read from a **gitignored `.env`** — never hardcoded, never committed.
- The raw pull is written to **`data/raw/`**, which is **gitignored** — the private tier.
- Nothing here is publish-safe. Anonymization and aggregation happen downstream
  (`02_clean_audit.ipynb` / `03_eda.ipynb`) before anything can go public.

**Before running:** copy `.env.example` to `.env` and fill in your PostHog key, host,
and project ID.


In [27]:
# Dependencies (run once). Safe to re-run.
%pip install requests pandas pyarrow python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [28]:
import os
import json
from pathlib import Path
from datetime import datetime, timedelta, timezone

import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # reads .env from the repo root

POSTHOG_API_KEY    = os.environ.get("POSTHOG_API_KEY")
POSTHOG_HOST       = os.environ.get("POSTHOG_HOST", "https://us.posthog.com").rstrip("/")
POSTHOG_PROJECT_ID = os.environ.get("POSTHOG_PROJECT_ID")

assert POSTHOG_API_KEY,    "POSTHOG_API_KEY missing — copy .env.example to .env and fill it in."
assert POSTHOG_PROJECT_ID, "POSTHOG_PROJECT_ID missing — set it in .env."

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Host:    {POSTHOG_HOST}")
print(f"Project: {POSTHOG_PROJECT_ID}")
print(f"Raw dir: {RAW_DIR.resolve()}")

Host:    https://us.posthog.com
Project: 479420
Raw dir: C:\Users\Faith\Desktop\peras-analytics\peras-analytics\data\raw


## Helper: run a HogQL query

One small function we'll reuse everywhere. It POSTs a HogQL query to PostHog's
query endpoint and returns a tidy `DataFrame`. HogQL is PostHog's SQL dialect
(ClickHouse-based) — this is where you get to show real SQL against a live
product-analytics warehouse.


In [29]:
def run_hogql(query: str, timeout: int = 120) -> pd.DataFrame:
    """Execute a HogQL query against the PostHog Query API and return a DataFrame."""
    url = f"{POSTHOG_HOST}/api/projects/{POSTHOG_PROJECT_ID}/query/"
    headers = {
        "Authorization": f"Bearer {POSTHOG_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {"query": {"kind": "HogQLQuery", "query": query}}

    resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
    if resp.status_code != 200:
        # Surface PostHog's error body — it's usually specific and helpful.
        raise RuntimeError(f"PostHog {resp.status_code}: {resp.text[:800]}")

    data = resp.json()
    cols = [c if isinstance(c, str) else c.get("name") for c in data.get("columns", [])]
    return pd.DataFrame(data.get("results", []), columns=cols)


# Smoke test: confirm auth + connectivity with a trivial query.
run_hogql("SELECT 1 AS ok")

,ok
0,1


## Step 1 · Event inventory

Before pulling anything heavy, see what events exist and how common they are.
This doubles as the start of your **data dictionary** (a Phase 1 deliverable) and
tells you which events are worth scoping into the real pull.

> Adjust the lookback window to match how long Peras has been tracking.


In [30]:
LOOKBACK_DAYS = 90

event_inventory = run_hogql(f"""
    SELECT
        event                       AS event_name,
        count()                     AS event_count,
        count(DISTINCT distinct_id) AS unique_users,
        min(timestamp)              AS first_seen,
        max(timestamp)              AS last_seen
    FROM events
    WHERE timestamp >= now() - INTERVAL {LOOKBACK_DAYS} DAY
    GROUP BY event
    ORDER BY event_count DESC
    LIMIT 1000
""")

if len(event_inventory) == 1000:
    print("WARNING: hit the LIMIT -- there are 1000+ distinct event names, raise the limit above.")

event_inventory


,event_name,event_count,unique_users,first_seen,last_seen
0,$pageview,14871,3593,2026-06-21T11:50:05.496000Z,2026-07-16T22:28:29.147000Z
1,$feature_flag_called,8053,3095,2026-06-22T05:08:09.435000Z,2026-07-16T21:12:59.636000Z
2,app_boot_started,7109,3575,2026-06-21T11:50:05.430000Z,2026-07-16T22:28:29.269000Z
3,app_root_rendered,6593,3554,2026-06-21T11:50:05.501000Z,2026-07-16T22:28:29.278000Z
4,landing_cta_impression,6116,1943,2026-06-21T12:08:47.505000Z,2026-07-16T22:28:29.983000Z
...,...,...,...,...,...
169,$conversation_message_received,1,1,2026-07-16T06:36:36.992000Z,2026-07-16T06:36:36.992000Z
170,sample_full_reader_clicked,1,1,2026-07-14T16:14:07.389000Z,2026-07-14T16:14:07.389000Z
171,$conversation_ticket_created,1,1,2026-07-16T06:36:36.917000Z,2026-07-16T06:36:36.917000Z
172,public_surface_error,1,1,2026-07-10T12:01:40.259000Z,2026-07-10T12:01:40.259000Z


## Step 2 · Property + PII audit

For a couple of the important events, look at *what properties they carry*. This is
where you spot **PII** (emails, names, IPs, `$geoip_*`) so you know what to drop or
hash at extraction, and it feeds the data dictionary. Note anything sensitive here —
it directly informs the safety protocol.

> Change `'$pageview'` / `'textbook_generated'` to real event names from Step 1.


In [31]:
# Peek at the property keys present on a sample of one event type.
sample = run_hogql("""
    SELECT properties
    FROM events
    WHERE event = 'create_reader_view'          -- <-- replace with a real Peras event
    ORDER BY timestamp DESC
    LIMIT 25
""")

# Flatten the JSON property blobs to see which keys appear.
prop_keys = {}
for blob in sample["properties"].dropna():
    d = blob if isinstance(blob, dict) else json.loads(blob)
    for k in d:
        prop_keys[k] = prop_keys.get(k, 0) + 1

with pd.option_context("display.max_rows", None):
    display(pd.Series(prop_keys).sort_values(ascending=False).to_frame("appears_in_n_rows"))


,appears_in_n_rows
$active_feature_flags,25
$autocapture_disabled_server_side,25
$browser,25
$browser_language,25
$browser_language_prefix,25
$browser_version,25
$config_defaults,25
$configured_session_timeout_ms,25
$current_url,25
$dead_clicks_enabled_server_side,25


### PII checklist (filled in from Step 2 audit against `$pageview`, 2026-07-15)

| Field | Where it lives | Identifying? | Handling |
|---|---|---|---|
| `distinct_id` | every event | yes (often email pre-`identify()`) | drop from public tier; hash for private-tier user-level joins |
| `$ip` | event props | yes (raw IP) | drop entirely, never pull |
| `$raw_user_agent` | event props | quasi (device fingerprint) | drop; `$browser`/`$os` cover what we need |
| `$geoip_latitude`, `$geoip_longitude` | event props | yes (precise coordinates) | drop entirely |
| `$geoip_city_name` | event props | quasi (city-level) | drop; keep only `$geoip_country_code` |
| `$geoip_postal_code` | event props | yes (address-level) | drop entirely |
| `$geoip_subdivision_1/2_name`, `_code` | event props | quasi (state/county-level) | drop entirely |
| `$user_id` | event props | yes | drop / never pull unless needed |
| `$device_id`, `$window_id`, `$pageview_id`, `$insert_id`, `$prev_pageview_id` | event props | quasi (re-identifies a browsing session) | drop; not needed for aggregate funnel analysis |
| `$session_id` | event props | quasi (re-identifies a browsing session) | **keep in the private raw tier** -- needed for session definition in 02_clean_audit; still drop before anything reaches the public/aggregated tier |
| Ad-click IDs: `gclid`, `gbraid`, `wbraid`, `fbclid`, `msclkid`, `ttclid`, `twclid`, `dclid`, `li_fat_id`, `irclid`, `igshid`, `epik`, `gclsrc`, `sccid`, `qclid`, `rdt_cid`, `mc_cid`, `_kx`, `gad_source` | event props, **and embedded as query params in `$current_url`** | yes (joinable back to a person via the ad platform) | drop the properties; **strip the query string from `$current_url` at extraction** (see Step 3 -- `$current_url` alone was leaking these back in) |
| `$referrer`, `$session_entry_referrer` | event props | quasi (may embed auth tokens in query string) | drop |
| `title` | event props | low/quasi (may leak unreleased page/content names) | exclude until reviewed for a public artifact |
| `$active_feature_flags`, `$feature_flag_payloads`, `$feature/*`, `$sdk_debug_*`, `$lib_*`, `$config_defaults`, `$transformations_succeeded` | event props | no (internal instrumentation noise) | exclude -- not sensitive, just irrelevant |

**Already coarse enough to keep:** `$geoip_country_code`/`$geoip_country_name`, `$browser`, `$os`, `$referring_domain` (domain only, unlike `$referrer`), `utm_source`/`utm_medium`/`utm_campaign`/`utm_term`.

**Not PII -- internal business IDs, safe to keep raw:** `account_id`, `project_id`, `user_id` (Peras's own, distinct from PostHog's `$user_id`), `environment`, `$virt_is_bot`, `$virt_traffic_type` (bot/traffic classification -- needed to filter bot traffic before trusting any funnel metric).

**Minimize at extraction:** only pull identifiers you actually need for the analysis.


## Step 3 - Scoped extraction

Now pull the real working dataset. Keep it to the events and window you actually
need. We select explicit columns (not `SELECT *`) so we're deliberate about what
leaves PostHog -- including *not* pulling raw IPs/emails.

**Pulled in weekly batches** (see `BATCH_DAYS` below) rather than one query across
the full `PULL_DAYS` window. This avoids the Query API's per-query row cap and keeps
memory bounded regardless of how much `PULL_DAYS` grows later -- see the "note on
large pulls" after this pull for why.

Land the result to `data/raw/` as **parquet** (compact, typed, fast). This folder is
gitignored -- it never gets committed.


In [32]:
PULL_DAYS = 90
BATCH_DAYS = 7  # pull in weekly windows -- see the note above and after this cell
BATCH_ROW_LIMIT = 200_000  # explicit cap per batch -- PostHog silently truncates to 100
                            # rows if a query has NO limit clause at all. This is what was
                            # actually causing the "only 15 people" undercount, not the
                            # 90-day window -- see the note after this cell.

# Column list driven by the acquisition / activation / engagement / retention / experiments
# questions -- see the PII checklist above for what stays excluded and why.
# current_url is stripped to the path (before '?') so ad-click IDs (gclid, fbclid, etc.)
# embedded as query params don't leak back in.
COLUMNS_SQL = """
        -- Core
        event,
        timestamp,
        distinct_id,
        person_id,                                    -- canonical unique-user id (anon -> identified safe)
        properties.account_id           AS account_id, -- Peras's own stable account id
        properties.project_id           AS project_id, -- links project/generation events to one textbook

        -- Acquisition
        splitByChar('?', coalesce(properties.$current_url, ''))[1] AS current_url,
        properties.$browser             AS browser,
        properties.$os                  AS os,
        properties.$geoip_country_code  AS country,
        properties.$referring_domain    AS referring_domain,
        properties.entry_path           AS entry_path,
        properties.auth_provider        AS auth_provider,
        properties.utm_source           AS utm_source,
        properties.utm_medium           AS utm_medium,
        properties.utm_campaign         AS utm_campaign,
        properties.utm_term             AS utm_term,

        -- Experiments (three separate tests Peras runs)
        properties.landing_experiment          AS landing_experiment,
        properties.landing_variant             AS landing_variant,
        properties.campaign_intent             AS campaign_intent,
        properties.campaign_variant            AS campaign_variant,
        properties.sample_consent_experiment   AS sample_consent_experiment,
        properties.sample_consent_variant      AS sample_consent_variant,

        -- Activation / engagement depth
        properties.generation_mode         AS generation_mode,
        properties.reader_mode             AS reader_mode,
        properties.reader_section_count    AS reader_section_count,
        properties.reader_written_sections AS reader_written_sections,
        properties.has_paced_reader        AS has_paced_reader,

        -- Data quality (bot/staging filtering happens in 02_clean_audit)
        properties.environment       AS environment,
        properties.$virt_is_bot      AS is_bot,
        properties.$virt_traffic_type AS traffic_type,

        -- Session definition (private raw tier only -- see PII checklist)
        properties.$session_id AS session_id
"""

pull_end = datetime.now(timezone.utc)
pull_start = pull_end - timedelta(days=PULL_DAYS)

# Build weekly (or BATCH_DAYS-sized) windows covering [pull_start, pull_end).
windows = []
cursor = pull_start
while cursor < pull_end:
    window_end = min(cursor + timedelta(days=BATCH_DAYS), pull_end)
    windows.append((cursor, window_end))
    cursor = window_end

print(f"Pulling {PULL_DAYS} days in {len(windows)} batch(es) of up to {BATCH_DAYS} days each.")

batches = []
for i, (start, end) in enumerate(windows, start=1):
    start_str = start.strftime("%Y-%m-%d %H:%M:%S")
    end_str = end.strftime("%Y-%m-%d %H:%M:%S")
    batch = run_hogql(f"""
        SELECT
{COLUMNS_SQL}
        FROM events
        WHERE timestamp >= toDateTime('{start_str}')
          AND timestamp <  toDateTime('{end_str}')
        ORDER BY timestamp
        LIMIT {BATCH_ROW_LIMIT}
    """)
    if len(batch) == BATCH_ROW_LIMIT:
        print(f"  WARNING: batch {i} hit BATCH_ROW_LIMIT ({BATCH_ROW_LIMIT:,}) -- likely truncated. "
              f"Shrink BATCH_DAYS or raise BATCH_ROW_LIMIT.")
    print(f"  Batch {i}/{len(windows)} ({start_str} -> {end_str}): {len(batch):,} rows")
    batches.append(batch)

events_raw = pd.concat(batches, ignore_index=True)
print(f"Pulled {len(events_raw):,} rows across "
      f"{events_raw['event'].nunique()} event types across {len(windows)} batches.")
# Shape/columns only -- never preview raw rows here. This notebook (and its saved
# outputs) gets committed even though data/raw/ itself is gitignored.
print(f"Shape: {events_raw.shape}")
print(f"Columns: {list(events_raw.columns)}")


Pulling 90 days in 13 batch(es) of up to 7 days each.
  Batch 1/13 (2026-04-17 22:30:52 -> 2026-04-24 22:30:52): 0 rows
  Batch 2/13 (2026-04-24 22:30:52 -> 2026-05-01 22:30:52): 0 rows
  Batch 3/13 (2026-05-01 22:30:52 -> 2026-05-08 22:30:52): 0 rows
  Batch 4/13 (2026-05-08 22:30:52 -> 2026-05-15 22:30:52): 0 rows
  Batch 5/13 (2026-05-15 22:30:52 -> 2026-05-22 22:30:52): 0 rows
  Batch 6/13 (2026-05-22 22:30:52 -> 2026-05-29 22:30:52): 0 rows
  Batch 7/13 (2026-05-29 22:30:52 -> 2026-06-05 22:30:52): 0 rows
  Batch 8/13 (2026-06-05 22:30:52 -> 2026-06-12 22:30:52): 0 rows
  Batch 9/13 (2026-06-12 22:30:52 -> 2026-06-19 22:30:52): 0 rows
  Batch 10/13 (2026-06-19 22:30:52 -> 2026-06-26 22:30:52): 9,150 rows
  Batch 11/13 (2026-06-26 22:30:52 -> 2026-07-03 22:30:52): 33,356 rows
  Batch 12/13 (2026-07-03 22:30:52 -> 2026-07-10 22:30:52): 50,000 rows
  Batch 13/13 (2026-07-10 22:30:52 -> 2026-07-16 22:30:52): 18,958 rows
Pulled 111,464 rows across 174 event types across 13 batches.
Sha

In [33]:
out_path = RAW_DIR / "events_raw.parquet"
events_raw.to_parquet(out_path, index=False)
print(f"Wrote {len(events_raw):,} rows -> {out_path.resolve()}")
print("Reminder: data/raw/ is gitignored. This file stays private.")

Wrote 111,464 rows -> C:\Users\Faith\Desktop\peras-analytics\peras-analytics\data\raw\events_raw.parquet
Reminder: data/raw/ is gitignored. This file stays private.


### A note on large pulls

Batching is implemented above: the pull loops over `BATCH_DAYS`-sized windows and
concats the results, rather than issuing one query across the full `PULL_DAYS` range.

**The actual bug that caused the "only 15 people, but the business sees hundreds
daily" undercount wasn't the time window -- it was a missing `LIMIT`.** PostHog's
Query API silently truncates any HogQL query with no `LIMIT` clause to 100 rows, no
error. Step 1's event inventory proved this directly: it returned exactly 100 rows
even though the project has 165 distinct event types. Step 3 had the same bug, and
it was worse there -- each of the 13 weekly batches was independently truncated to
its first ~100 rows by timestamp, so the extract only ever captured whoever
happened to show up earliest in each week, not a representative sample.

Both queries now have an explicit `LIMIT` (`1000` for the event inventory, tunable
`BATCH_ROW_LIMIT` for the batched pull) plus a check that warns if a batch actually
hits the limit -- if that ever fires, shrink `BATCH_DAYS` (smaller windows, same
total range) rather than raising the limit indefinitely.

---

## Next

-> **`02_clean_audit.ipynb`** -- dedupe, handle nulls/types, strip bot & internal
traffic (`is_bot`, `traffic_type`, `environment` are now in the extract for this),
define sessions (`session_id` is now in the extract for this), and write up a short
data-quality report. That notebook is where the raw tier gets turned into something
trustworthy; aggregation for the public tier happens after that.
